In [1]:
!pip install wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 36.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.4/468.4 kB 27.3 MB/s eta 0:00:00


In [4]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, losses, metrics
import matplotlib.pyplot as plt
from PIL import Image
import time

import wandb


In [5]:
IMAGE_SIZE = 128
CHANNELS = 3

BATCH_SIZE = 64
Z_DIM = 100
EPOCHS_GRID_SEARCH = 15  #fast search on small dataset


# Optimizer settings
LEARNING_RATE_G = 0.0002  
LEARNING_RATE_D = 0.0002  
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.999


D_LR_DECAY_FACTOR = 0.95  
D_LR_DECAY_EVERY = 5       
D_LR_MIN = 0.00005        

NOISE_PARAM = 0.05

DATA_ROOT = "fish-dataset"
OUT_DIR = "outputs_keras"
CKPT_DIR = "checkpoints_keras"

PATIENCE = 10
MIN_DELTA = 0.01

USE_WANDB = True
WANDB_PROJECT = "fish-dcgan"

# Grid search settings
GRID_SEARCH_IMAGES = 100  

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

In [6]:
def make_dataset(num_images=None):
    ds = keras.utils.image_dataset_from_directory(
        DATA_ROOT,
        label_mode=None,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=42,
    )
    
    norm = layers.Rescaling(scale=1.0 / 127.5, offset=-1.0)
    ds = ds.map(lambda x: norm(tf.cast(x, tf.float32)), num_parallel_calls=tf.data.AUTOTUNE)
    
    if num_images is not None:
        num_batches = (num_images + BATCH_SIZE - 1) // BATCH_SIZE
        ds = ds.take(num_batches)
        print(f" mini-dataset: {num_images} images ({num_batches} batches)")
    
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds_mini = make_dataset(num_images=GRID_SEARCH_IMAGES)

Found 3983 files.
 mini-dataset: 100 images (2 batches)


In [7]:
NGF = 64

def build_generator(z_dim=Z_DIM, ngf=NGF):
    inp = layers.Input(shape=(z_dim,))
    x = layers.Reshape((1, 1, z_dim))(inp)

    x = layers.Conv2DTranspose(ngf * 8, 4, strides=1, padding="valid", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(ngf * 4, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(ngf * 2, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(ngf, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(ngf // 2, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    out = layers.Conv2DTranspose(
        CHANNELS, 4, strides=2, padding="same", use_bias=False, activation="tanh"
    )(x)

    return models.Model(inp, out, name="generator")

In [8]:
NDF = 64

def build_discriminator(img_shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS), ndf=NDF):
    inp = layers.Input(shape=img_shape)

    x = layers.Conv2D(ndf, 4, strides=2, padding="same", use_bias=False)(inp)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(ndf * 2, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(ndf * 4, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(ndf * 8, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(ndf * 8, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(1, 4, strides=1, padding="valid", use_bias=False)(x)
    out = layers.Flatten()(x)

    return models.Model(inp, out, name="discriminator")

In [9]:
class DCGAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim, noise_param=0.05):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.noise_param = noise_param
        self.loss_fn = losses.BinaryCrossentropy(from_logits=True)
        
        self.current_d_lr = LEARNING_RATE_D

    def compile(self, d_optimizer, g_optimizer):
        super().compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.d_loss_metric = metrics.Mean(name="d_loss")
        self.g_loss_metric = metrics.Mean(name="g_loss")

    @property
    def metrics(self):
        return [self.d_loss_metric, self.g_loss_metric]

    def train_step(self, real_images):
        if isinstance(real_images, tuple):
            real_images = real_images[0]

        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))

        with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
            generated_images = self.generator(random_latent_vectors, training=True)

            real_predictions = self.discriminator(real_images, training=True)
            fake_predictions = self.discriminator(generated_images, training=True)

            real_labels = tf.ones_like(real_predictions)
            fake_labels = tf.zeros_like(fake_predictions)

            real_noisy_labels = real_labels - self.noise_param * tf.random.uniform(
                tf.shape(real_predictions), 0.0, 1.0
            )
            fake_noisy_labels = fake_labels + self.noise_param * tf.random.uniform(
                tf.shape(fake_predictions), 0.0, 1.0
            )

            d_real_loss = self.loss_fn(real_noisy_labels, real_predictions)
            d_fake_loss = self.loss_fn(fake_noisy_labels, fake_predictions)
            d_loss = (d_real_loss + d_fake_loss) / 2.0

            g_loss = self.loss_fn(tf.ones_like(fake_predictions), fake_predictions)

        grad_d = disc_tape.gradient(d_loss, self.discriminator.trainable_variables)
        grad_g = gen_tape.gradient(g_loss, self.generator.trainable_variables)

        self.d_optimizer.apply_gradients(zip(grad_d, self.discriminator.trainable_variables))
        self.g_optimizer.apply_gradients(zip(grad_g, self.generator.trainable_variables))

        self.d_loss_metric.update_state(d_loss)
        self.g_loss_metric.update_state(g_loss)

        return {m.name: m.result() for m in self.metrics}

In [10]:
#reduce discriminator learning rate so it doesn't get too strong
    
class DiscriminatorLRDecay(keras.callbacks.Callback):
    def __init__(self, decay_factor=0.95, decay_every=5, min_lr=0.00005):
        super().__init__()
        self.decay_factor = decay_factor
        self.decay_every = decay_every
        self.min_lr = min_lr
    
    def on_epoch_end(self, epoch, logs=None):

        if (epoch + 1) % self.decay_every == 0:
            old_lr = self.model.d_optimizer.learning_rate.numpy()
            new_lr = max(old_lr * self.decay_factor, self.min_lr)
            self.model.d_optimizer.learning_rate.assign(new_lr)
            self.model.current_d_lr = new_lr
            print(f"\nDiscriminator LR: {old_lr:.6f} → {new_lr:.6f}")


class WandbImageCallback(keras.callbacks.Callback):
    
    def __init__(self, latent_dim, num_samples=16):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_samples = num_samples
        self.fixed_latent = tf.random.normal(shape=(num_samples, latent_dim))
    
    def denorm_to_01(self, x):
        return (x + 1.0) / 2.0
    
    def on_epoch_end(self, epoch, logs=None):
        fake = self.model.generator(self.fixed_latent, training=False)
        fake_01 = tf.clip_by_value(self.denorm_to_01(fake), 0.0, 1.0)
        
        fig, axes = plt.subplots(4, 4, figsize=(8, 8))
        for i, ax in enumerate(axes.flat):
            ax.imshow(fake_01[i].numpy())
            ax.axis("off")
        plt.suptitle(f"Epoch {epoch + 1}")
        plt.tight_layout()
        
        if USE_WANDB:
            wandb.log({"samples": wandb.Image(fig)}, step=epoch)
        
        plt.close(fig)


class EarlyStoppingCallback(keras.callbacks.Callback):
    
    def __init__(self, patience=10, min_delta=0.01):
        super().__init__()
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.wait = 0
    
    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('g_loss')
        if current_loss is None:
            return
        
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.wait = 0
            self.model.generator.save_weights(os.path.join(CKPT_DIR, "g_best.weights.h5"))
            self.model.discriminator.save_weights(os.path.join(CKPT_DIR, "d_best.weights.h5"))
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                print(f"\nearly stopping at epoch {epoch + 1}. Best G loss: {self.best_loss:.4f}")

In [11]:
def train_dcgan(dataset, epochs, lr_g, lr_d, noise_param, use_wandb=True, run_name=None):
    if use_wandb:
        wandb.init(
            project=WANDB_PROJECT,
            name=run_name,
            config={
                "batch_size": BATCH_SIZE,
                "z_dim": Z_DIM,
                "lr_g": lr_g,
                "lr_d": lr_d,
                "noise_param": noise_param,
                "epochs": epochs,
                "d_lr_decay_factor": D_LR_DECAY_FACTOR,
                "d_lr_decay_every": D_LR_DECAY_EVERY,
            }
        )
    
    gen = build_generator(z_dim=Z_DIM)
    disc = build_discriminator()
    
    dcgan = DCGAN(discriminator=disc, generator=gen, latent_dim=Z_DIM, noise_param=noise_param)
    
    dcgan.compile(
        d_optimizer=optimizers.Adam(learning_rate=lr_d, beta_1=ADAM_BETA_1, beta_2=ADAM_BETA_2),
        g_optimizer=optimizers.Adam(learning_rate=lr_g, beta_1=ADAM_BETA_1, beta_2=ADAM_BETA_2),
    )
    
    callbacks = [
        DiscriminatorLRDecay(
            decay_factor=D_LR_DECAY_FACTOR,
            decay_every=D_LR_DECAY_EVERY,
            min_lr=D_LR_MIN
        ),
        EarlyStoppingCallback(patience=PATIENCE, min_delta=MIN_DELTA)
    ]
    
    if use_wandb:
        callbacks.append(WandbImageCallback(latent_dim=Z_DIM))
        callbacks.append(keras.callbacks.LambdaCallback(
            on_epoch_end=lambda epoch, logs: wandb.log({
                "d_loss": logs['d_loss'],
                "g_loss": logs['g_loss'],
                "d_lr": dcgan.current_d_lr,
                "epoch": epoch
            })
        ))
    
    start_time = time.time()
    history = dcgan.fit(dataset, epochs=epochs, callbacks=callbacks)
    train_time = time.time() - start_time
    
    best_g_loss = min(history.history['g_loss'])
    
    if use_wandb:
        wandb.log({"train_time_seconds": train_time, "best_g_loss": best_g_loss})
        wandb.finish()
    
    return dcgan, history, best_g_loss, train_time

In [12]:
param_grid = {
    'lr_g': [0.0002],  
    'lr_d': [0.0001, 0.0002, 0.0003],  
    'noise_param': [0.0, 0.05, 0.1],  
}

print("grid search on 100 images")
print(f"total combinations: {len(param_grid['lr_d']) * len(param_grid['noise_param'])}")

results = []

for lr_d in param_grid['lr_d']:
    for noise in param_grid['noise_param']:
        lr_g = param_grid['lr_g'][0]
        
        print(f"Testing: G_LR={lr_g}, D_LR={lr_d}, Noise={noise}")
        
        try:
            run_name = f"grid_search_dlr{lr_d}_noise{noise}"
            dcgan, history, best_g_loss, train_time = train_dcgan(
                dataset=train_ds_mini,
                epochs=EPOCHS_GRID_SEARCH,
                lr_g=lr_g,
                lr_d=lr_d,
                noise_param=noise,
                use_wandb=USE_WANDB,
                run_name=run_name
            )
            
            results.append({
                'lr_g': lr_g,
                'lr_d': lr_d,
                'noise': noise,
                'best_g_loss': best_g_loss,
                'train_time': train_time
            })
            
            print(f"\ncompleted in {train_time:.1f}s, best G loss: {best_g_loss:.4f}")
            
        except Exception as e:
            print(f"✗ failed with error: {e}")
            continue

best_result = min(results, key=lambda x: x['best_g_loss'])

print("best hyperparams:")
print("*"*70)
print(f"generator lr:     {best_result['lr_g']}")
print(f"discriminator lr: {best_result['lr_d']}")
print(f"noise parameter:  {best_result['noise']}")
print(f"best g loss:      {best_result['best_g_loss']:.4f}")
print(f"training time:    {best_result['train_time']:.1f}s")

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

grid search on 100 images
total combinations: 9
Testing: G_LR=0.0002, D_LR=0.0001, Noise=0.0


wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: ERROR Invalid API key: API key must have 40+ characters, has 1.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wa

Epoch 1/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - d_loss: 0.5783 - g_loss: 1.0339
Epoch 2/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.2223 - g_loss: 2.0883
Epoch 3/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.1797 - g_loss: 2.3076
Epoch 4/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.1781 - g_loss: 2.2363
Epoch 5/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - d_loss: 0.1646 - g_loss: 2.3367
Discriminator LR: 0.000100 → 0.000095
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.1451 - g_loss: 3.0770
Epoch 6/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.2380 - g_loss: 3.9025
Epoch 7/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.6077 - g_loss: 1.8853
Epoch 8/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.1243 - g_loss: 12.6094
Epoch 9/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 1.0622 - g_loss: 2.5649
Epoch 10/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - d_loss: 0.0641 - g_loss: 12.6012
Discriminator LR: 0.000095 → 0.000090
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_l

best_g_loss,▁
d_loss,▅▂▂▂▁▂▅▁█▁▁
d_lr,████▁▁▁▁▁▁▁
epoch,▁▂▂▃▄▅▅▆▇▇█
g_loss,▁▂▂▂▂▃▁█▂█▃
train_time_seconds,▁
best_g_loss,1.03388
d_loss,0.12692
d_lr,9e-05
epoch,10
g_loss,5.10197


...

best_g_loss,▁
d_loss,▃▂▁█▂▂▁▇▃▂▄▂
d_lr,▁▁▁▁█████▇▇▇
epoch,▁▂▂▃▄▄▅▅▆▇▇█
g_loss,▂▁▃▃▅▂▃▄█▆▂▇
train_time_seconds,▁
best_g_loss,1.86837
d_loss,0.52426
d_lr,0.00027
epoch,11
g_loss,22.93365



completed in 43.2s, best G loss: 1.8684
Testing: G_LR=0.0002, D_LR=0.0003, Noise=0.1


Epoch 1/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - d_loss: 0.5804 - g_loss: 3.4137
Epoch 2/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.9861 - g_loss: 5.8674
Epoch 3/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.5635 - g_loss: 5.0850
Epoch 4/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.5384 - g_loss: 9.5536 
Epoch 5/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - d_loss: 2.0104 - g_loss: 4.9257
Discriminator LR: 0.000300 → 0.000285
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 1.6265 - g_loss: 9.7786
Epoch 6/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - d_loss: 0.8852 - g_loss: 18.8718
Epoch 7/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 1.3022 - g_loss: 3.4458
Epoch 8/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.7567 - g_loss: 18.9484
Epoch 9/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d_loss: 0.5199 - g_loss: 15.7563
Epoch 10/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - d_loss: 0.5250 - g_loss: 5.9059
Discriminator LR: 0.000285 → 0.000271
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - d

best_g_loss,▁
d_loss,▁▄▁▁█▃▆▂▁▂▂
d_lr,▁▁▁▁█████▇▇
epoch,▁▂▂▃▄▅▅▆▇▇█
g_loss,▁▂▂▄▄█▁█▇▁▇
train_time_seconds,▁
best_g_loss,3.41367
d_loss,0.66895
d_lr,0.00027
epoch,10
g_loss,16.66602



completed in 39.4s, best G loss: 3.4137
best hyperparams:
**********************************************************************
generator lr:     0.0002
discriminator lr: 0.0002
noise parameter:  0.1
best g loss:      0.7767
training time:    38.7s
